
# Street barrier detector — Colab training (YOLO11s)

Trains the 5-class detector (`curb_ramp`, `crosswalk`, `stairs_steps`,
`tactile_paving`, `ramp`) on a Colab **GPU** runtime and produces:

| payload path (in `weights_payload.zip`) | repo path |
|---|---|
| `best.pt` | `backend/ml/street/weights/best.pt` |
| `best.onnx` | `backend/ml/street/weights/best.onnx` |
| `street.json` | `backend/ml/street/metrics/street.json` |
| `samples/*.jpg` | `backend/ml/street/samples/` |

**Before running:** `Runtime -> Change runtime type -> GPU (T4/L4/A100)`.

Local-machine probe: 1 epoch on Apple MPS = ~774 s > ~240 s ⇒ Colab path
(machine rule from the task spec).

**Roboflow key:** add `ROBOFLOW_API_KEY` to Colab Secrets (Secrets sidebar), or
paste it when prompted. It is never inserted into outputs or saved into files
we commit.

Directory layout mirrors the repo (`backend/ml/street/...`) so the embedded
`build_dataset.py` runs unchanged.


In [ ]:

import os, getpass, pathlib

for d in ("data", "configs", "scripts", "weights", "samples"):
    pathlib.Path(f"backend/ml/street/{d}").mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    print("key: loaded from Colab Secrets")
except Exception:
    ROBOFLOW_API_KEY = getpass.getpass("ROBOFLOW_API_KEY: ")
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
print("key prefix:", ROBOFLOW_API_KEY[:3] + "***")  # masked, never echoed fully


In [ ]:

!pip -q install "ultralytics>=8.3" pyyaml imagehash
import ultralytics
print("ultralytics", ultralytics.__version__)


In [ ]:

import base64, pathlib
SCRIPT_B64 = {
    "build_dataset.py": "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCdWlsZCB0aGUgdW5pZmllZCBtbC9zdHJlZXQgWU9MTyBkYXRhc2V0LgoKUmVhZHMgdGhlIGZvdXIgcmF3IGRhdGFzZXRzICgzIFJvYm9mbG93IFlPTE92OCBleHBvcnRzICsgV2hlZWxtYXAgWmVub2RvLApjb252ZXJ0ZWQgb24gdGhlIGZseSBmcm9tIENWQVQgWE1MKSwgbWFwcyBjbGFzc2VzIHBlciBjb25maWdzL2NsYXNzZXMueWFtbCwKZGVkdXBsaWNhdGVzIG5lYXItaWRlbnRpY2FsIGltYWdlcyB3aXRoIHBlcmNlcHR1YWwgaGFzaGluZyAoQkVGT1JFIHNwbGl0KSwKdGhlbiB3cml0ZXMgYW4gODAvMTAvMTAgc3RyYXRpZmllZCBzcGxpdCAoYW5jaG9yID0gcmFyZXN0IGNsYXNzIGluIGltYWdlKQppbnRvIGRhdGEveW9sby97dHJhaW4sdmFsLHRlc3R9LCB3aXRoIHBlci1jbGFzcyBpbnN0YW5jZSBjb3VudHMgYW5kIGEKcmVwb3J0IGZsYWdnaW5nIGNsYXNzZXMgdW5kZXIgMTUwIHRyYWluIGluc3RhbmNlcy4KClVzYWdlOgogIGJhY2tlbmQvdGlsZTJuZXQvLnZlbnYvYmluL3B5dGhvbiBiYWNrZW5kL21sL3N0cmVldC9zY3JpcHRzL2J1aWxkX2RhdGFzZXQucHkgXAogICAgICAtLWNsYXNzZXMgYmFja2VuZC9tbC9zdHJlZXQvY29uZmlncy9jbGFzc2VzLnlhbWwgXAogICAgICAtLW91dCBiYWNrZW5kL21sL3N0cmVldC9kYXRhL3lvbG8KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY29sbGVjdGlvbnMKaW1wb3J0IGdsb2IKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGl0ZXJ0b29scwppbXBvcnQganNvbgppbXBvcnQgcGF0aGxpYgppbXBvcnQgcmFuZG9tCmltcG9ydCBzaHV0aWwKaW1wb3J0IHhtbC5ldHJlZS5FbGVtZW50VHJlZSBhcyBFVAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB5YW1sCmZyb20gUElMIGltcG9ydCBJbWFnZQoKUk9CT0ZMT1dfRElSUyA9ICgic2lkZXdhbGtzaWRld2Fsa192MTAiLCAibG9nYW5faW50ZXJzZWN0aW9uc192MSIsICJjb21iaW5lZF9wcm9qX3YyIikKU1BMSVQgPSAoInRyYWluIiwgInZhbGlkIiwgInRlc3QiKQoKUElMX0hBU0hfS1cgPSBkaWN0KGhhc2hfc2l6ZT04LCBoaWdoZnJlcV9mYWN0b3I9NCkgICMgNjQtYml0IHBoYXNoIGZvciBzcGVlZAp0cnk6CiAgICBpbXBvcnQgaW1hZ2VoYXNoCgogICAgZGVmIHBoYXNoKHBhdGgpOgogICAgICAgIHdpdGggSW1hZ2Uub3BlbihwYXRoKSBhcyBpbToKICAgICAgICAgICAgcmV0dXJuIGltYWdlaGFzaC5waGFzaChpbS5jb252ZXJ0KCJMIiksICoqUElMX0hBU0hfS1cpCmV4Y2VwdCBJbXBvcnRFcnJvcjogICMgcHJhZ21hOiBubyBjb3ZlcgogICAgaW1hZ2VoYXNoID0gTm9uZQoKCmRlZiBtZDUocGF0aCk6CiAgICByZXR1cm4gaGFzaGxpYi5tZDUocGF0aGxpYi5QYXRoKHBhdGgpLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgbG9hZF9yb2JvZmxvdyhyb290OiBwYXRobGliLlBhdGgsIGRhdGFzZXQ6IHN0ciwgbWFwcGluZyk6CiAgICAiIiJkYXRhc2V0Wyd0cmFpbid8J3Rlc3QnfCd2YWxpZCddIHN0cnVjdHVyZSBmcm9tIGEgcm9ib2Zsb3cgZXhwb3J0LiIiIgogICAgb3V0ID0ge30KICAgIGltYWdlcyA9IFtdCiAgICBmb3Igc3AgaW4gU1BMSVQ6CiAgICAgICAgZCA9IHJvb3QgLyBkYXRhc2V0IC8gc3AKICAgICAgICBpbWFnZXMgKz0gc29ydGVkKChkIC8gImltYWdlcyIpLmdsb2IoIioiKSkKICAgIG5hbWVzID0geWFtbC5zYWZlX2xvYWQoKHJvb3QgLyBkYXRhc2V0IC8gImRhdGEueWFtbCIpLnJlYWRfdGV4dCgpKVsibmFtZXMiXQogICAgZm9yIGltZyBpbiBpbWFnZXM6CiAgICAgICAgbGJsID0gKHJvb3QgLyBkYXRhc2V0IC8gcGlja19zcGxpdChpbWcsIHJvb3QsIGRhdGFzZXQpIC8gImxhYmVscyIgLwogICAgICAgICAgICAgICAoaW1nLnN0ZW0gKyAiLnR4dCIpKQogICAgICAgIGJveGVzID0gW10KICAgICAgICB3LCBoID0gSW1hZ2Uub3BlbihpbWcpLnNpemUKICAgICAgICBpZiBsYmwuZXhpc3RzKCk6CiAgICAgICAgICAgIGZvciBsaW5lIGluIGxibC5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCk6CiAgICAgICAgICAgICAgICBwID0gbGluZS5zcGxpdCgpCiAgICAgICAgICAgICAgICBpZiBsZW4ocCkgPT0gNToKICAgICAgICAgICAgICAgICAgICBjbHNfbmF0aXZlID0gbmFtZXNbaW50KHBbMF0pXQogICAgICAgICAgICAgICAgICAgIGNsc190YXJnZXQgPSBtYXBwaW5nW2RhdGFzZXRdLmdldChjbHNfbmF0aXZlKQogICAgICAgICAgICAgICAgICAgIGlmIGNsc190YXJnZXQ6CiAgICAgICAgICAgICAgICAgICAgICAgICMgeHl3aCAtPiBwaXhlbCB4eXh5CiAgICAgICAgICAgICAgICAgICAgICAgIGN4LCBjeSwgYncsIGJoID0gKGZsb2F0KHYpIGZvciB2IGluIHBbMTpdKQogICAgICAgICAgICAgICAgICAgICAgICB4MSA9IChjeCAtIGJ3IC8gMikgKiB3CiAgICAgICAgICAgICAgICAgICAgICAgIHkxID0gKGN5IC0gYmggLyAyKSAqIGgKICAgICAgICAgICAgICAgICAgICAgICAgeDIgPSAoY3ggKyBidyAvIDIpICogdwogICAgICAgICAgICAgICAgICAgICAgICB5MiA9IChjeSArIGJoIC8gMikgKiBoCiAgICAgICAgICAgICAgICAgICAgICAgIGJveGVzLmFwcGVuZCgoY2xzX3RhcmdldCwgeDEsIHkxLCB4MiwgeTIpKQogICAgICAgIG91dFtpbWddID0gZGljdChzcmM9ZGF0YXNldCwgc2l6ZT0odywgaCksIGJveGVzPWJveGVzKQogICAgcmV0dXJuIG91dAoKCmRlZiBwaWNrX3NwbGl0KGltZywgcm9vdCwgZGF0YXNldCk6CiAgICBmb3Igc3AgaW4gU1BMSVQ6CiAgICAgICAgaWYgKHJvb3QgLyBkYXRhc2V0IC8gc3AgLyAiaW1hZ2VzIikuaXNfZGlyKCk6CiAgICAgICAgICAgIHEgPSByb290IC8gZGF0YXNldCAvIHNwIC8gImltYWdlcyIgLyBpbWcubmFtZQogICAgICAgICAgICBpZiBxLmV4aXN0cygpOgogICAgICAgICAgICAgICAgcmV0dXJuIHNwCiAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihpbWcpCgoKZGVmIGxvYWRfd2hlZWxtYXAoeG1sX3BhdGg6IHBhdGhsaWIuUGF0aCwgaW1nX2RpcjogcGF0aGxpYi5QYXRoLCBtYXBwaW5nKToKICAgIHJvb3QgPSBFVC5wYXJzZSh4bWxfcGF0aCkuZ2V0cm9vdCgpCiAgICBvdXQgPSB7fQogICAgbSA9IG1hcHBpbmdbIndoZWVsbWFwIl0KICAgIGZvciBpbSBpbiByb290LmZpbmRhbGwoImltYWdlIik6CiAgICAgICAgbmFtZSA9IGltLmF0dHJpYlsibmFtZSJdLnNwbGl0KCIvIilbLTFdCiAgICAgICAgcCA9IGltZ19kaXIgLyBuYW1lCiAgICAgICAgaWYgbm90IHAuZXhpc3RzKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgIyBza2lwIGltYWdlcyB0aGF0IG9ubHkgY29udGFpbiBkcm9wcGVkIChncmFiX2JhcikgYm94ZXMKICAgICAgICBib3hlcyA9IFtdCiAgICAgICAgZm9yIGIgaW4gaW0uZmluZGFsbCgiYm94Iik6CiAgICAgICAgICAgIGNsc190YXJnZXQgPSBtLmdldChiLmF0dHJpYlsibGFiZWwiXSkKICAgICAgICAgICAgaWYgY2xzX3RhcmdldDoKICAgICAgICAgICAgICAgIGJveGVzLmFwcGVuZCgoY2xzX3RhcmdldCwgZmxvYXQoYi5hdHRyaWJbInh0bCJdKSwgZmxvYXQoYi5hdHRyaWJbInl0bCJdKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZmxvYXQoYi5hdHRyaWJbInhiciJdKSwgZmxvYXQoYi5hdHRyaWJbInliciJdKSkpCiAgICAgICAgaWYgbm90IGJveGVzOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG91dFtwXSA9IGRpY3Qoc3JjPSJ3aGVlbG1hcCIsIHNpemU9KGludChpbS5hdHRyaWJbIndpZHRoIl0pLCBpbnQoaW0uYXR0cmliWyJoZWlnaHQiXSkpLAogICAgICAgICAgICAgICAgICAgICAgYm94ZXM9Ym94ZXMpCiAgICByZXR1cm4gb3V0CgoKZGVmIGJ1aWxkX21hcHBpbmcoY2xhc3Nlc19jZmcpOgogICAgbSA9IGNvbGxlY3Rpb25zLmRlZmF1bHRkaWN0KGRpY3QpCiAgICB0YXJnZXRfbmFtZXMgPSBbY1sibmFtZSJdIGZvciBjIGluIGNsYXNzZXNfY2ZnWyJjbGFzc2VzIl1dCiAgICBmb3IgYyBpbiBjbGFzc2VzX2NmZ1siY2xhc3NlcyJdOgogICAgICAgIGZvciBkcywgc3JjYyBpbiBjWyJzb3VyY2VzIl0uaXRlbXMoKToKICAgICAgICAgICAgZm9yIHMgaW4gKFtzcmNjXSBpZiBpc2luc3RhbmNlKHNyY2MsIHN0cikgZWxzZSBzcmNjKToKICAgICAgICAgICAgICAgIG1bZHNdW3NdID0gY1sibmFtZSJdCiAgICByZXR1cm4gZGljdChtKSwgdGFyZ2V0X25hbWVzCgoKZGVmIGRlZHVwZShpbmRleCwgaGFzaF90aHJlc2g9NCk6CiAgICAiIiJVbmlvbi1maW5kIGRlZHVwZSBieSBwaGFzaCBoYW1taW5nIGRpc3RhbmNlIDw9IGhhc2hfdGhyZXNoIChiZWZvcmUgc3BsaXQpLiIiIgogICAgaGFzaGVzLCBoYXNoZXNfdTE2ID0ge30sIFtdCiAgICBmb3IgcCBpbiBsaXN0KGluZGV4LmtleXMoKSk6CiAgICAgICAgaCA9IHBoYXNoKHN0cihwKSkKICAgICAgICBhcnIgPSBucC5hcnJheShoLmhhc2gpLmZsYXR0ZW4oKSAgIyAyNTYgYml0cwogICAgICAgIGhhc2hlc1twXSA9IGFycgogICAgTiA9IGxlbihoYXNoZXMpCiAgICBwYXRocyA9IGxpc3QoaGFzaGVzLmtleXMoKSkKICAgIGJpdHMgPSBucC5zdGFjayhbaGFzaGVzW3BdIGZvciBwIGluIHBhdGhzXSkgICMgKE4sIDI1NikgYm9vbAoKICAgIHBhcmVudCA9IGRpY3QoKHAsIHApIGZvciBwIGluIHBhdGhzKQoKICAgIGRlZiBmaW5kKHApOgogICAgICAgIHdoaWxlIHBhcmVudFtwXSAhPSBwOgogICAgICAgICAgICBwYXJlbnRbcF0gPSBwYXJlbnRbcGFyZW50W3BdXQogICAgICAgICAgICBwID0gcGFyZW50W3BdCiAgICAgICAgcmV0dXJuIHAKCiAgICBkZWYgdW5pb24oYSwgYik6CiAgICAgICAgcmEsIHJiID0gZmluZChhKSwgZmluZChiKQogICAgICAgIGlmIHJhICE9IHJiOgogICAgICAgICAgICBwYXJlbnRbcmJdID0gcmEKCiAgICBDSCA9IDUxMgogICAgZm9yIGkgaW4gcmFuZ2UoMCwgTiwgQ0gpOgogICAgICAgIHhiID0gYml0c1tpOmkgKyBDSF0gICMgKEMsMjU2KQogICAgICAgIGQgPSAobnAuY291bnRfbm9uemVybyh4Yls6LCBOb25lLCA6XSBeIGJpdHNbTm9uZSwgOiwgOl0sIGF4aXM9MikgPD0gaGFzaF90aHJlc2gpCiAgICAgICAgZm9yIHIsIGMgaW4gemlwKCpucC5ub256ZXJvKGQpKToKICAgICAgICAgICAgaWYgYyA+IGkgKyByOiAgIyBvbmx5IGZvcndhcmQgcGFpcnMgKHVwcGVyIHRyaWFuZ2xlIGJ5IGluZGV4KQogICAgICAgICAgICAgICAgdW5pb24ocGF0aHNbaSArIHJdLCBwYXRoc1tjXSkKICAgIGNsdXN0ZXJzID0gY29sbGVjdGlvbnMuZGVmYXVsdGRpY3QobGlzdCkKICAgIGZvciBwIGluIHBhdGhzOgogICAgICAgIGNsdXN0ZXJzW2ZpbmQocCldLmFwcGVuZChwKQogICAgcmV0dXJuIGNsdXN0ZXJzCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNsYXNzZXMiLCBkZWZhdWx0PSJiYWNrZW5kL21sL3N0cmVldC9jb25maWdzL2NsYXNzZXMueWFtbCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0iYmFja2VuZC9tbC9zdHJlZXQvZGF0YS95b2xvIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1oYXNoLXRocmVzaCIsIHR5cGU9aW50LCBkZWZhdWx0PTQpICAjIDY0LWJpdCBwaGFzaCBoYW1taW5nCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3ZlcnNhbXBsZS1taW4iLCB0eXBlPWludCwgZGVmYXVsdD0zMDAsCiAgICAgICAgICAgICAgICAgICAgaGVscD0iZHVwbGljYXRlIHRyYWluIGltYWdlcyBmb3IgcmFyZSBjbGFzc2VzICgwID0gb2ZmKSIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgY2ZnID0geWFtbC5zYWZlX2xvYWQob3BlbihhcmdzLmNsYXNzZXMpKQogICAgbWFwcGluZywgY2xzX25hbWVzID0gYnVpbGRfbWFwcGluZyhjZmcpCiAgICByb290ID0gcGF0aGxpYi5QYXRoKCJiYWNrZW5kL21sL3N0cmVldC9kYXRhIikKCiAgICBpbmRleCA9IHt9CiAgICBmb3IgZHMgaW4gUk9CT0ZMT1dfRElSUzoKICAgICAgICBpbmRleC51cGRhdGUobG9hZF9yb2JvZmxvdyhyb290LCBkcywgbWFwcGluZykpCiAgICB3bSA9IHJvb3QgLyAid2hlZWxtYXAiCiAgICBpbmRleC51cGRhdGUobG9hZF93aGVlbG1hcCh3bSAvICJ3bV9hbm5vdGF0aW9ucy54bWwiLCB3bSAvICJpbWFnZXMiLCBtYXBwaW5nKSkKCiAgICAjIGV4YWN0IGR1cGVzIGJ5IGNvbnRlbnQgaGFzaCBmaXJzdCAoUm9ib2Zsb3cgZXhwb3J0cyBtYXkgcmVwZWF0IGltYWdlcykKICAgIGltYWdlc19yYXcgPSBsZW4oaW5kZXgpCiAgICBzZWVuX21kNSA9IHt9CiAgICBmb3IgcCBpbiBsaXN0KGluZGV4LmtleXMoKSk6CiAgICAgICAgaCA9IG1kNShwKQogICAgICAgIGlmIGggaW4gc2Vlbl9tZDU6CiAgICAgICAgICAgIGRlbCBpbmRleFtwXQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlZW5fbWQ1W2hdID0gcAoKICAgIGNsdXN0ZXJzID0gZGVkdXBlKGluZGV4LCBhcmdzLmhhc2hfdGhyZXNoKQogICAga2VlcCA9IFtdCiAgICBkdXBzID0gMAogICAgZm9yIF8sIG1lbWJlcnMgaW4gY2x1c3RlcnMuaXRlbXMoKToKICAgICAgICBrZWVwLmFwcGVuZChtZW1iZXJzWzBdKQogICAgICAgIGR1cHMgKz0gbGVuKG1lbWJlcnMpIC0gMQoKICAgICMgaW5zdGFuY2UgY291bnRzIHBlciAoc3JjLCB0YXJnZXQtY2xhc3MpCiAgICBpbnN0ID0gY29sbGVjdGlvbnMuZGVmYXVsdGRpY3QoY29sbGVjdGlvbnMuQ291bnRlcikKICAgIGZvciBwIGluIGluZGV4OgogICAgICAgIGZvciBiIGluIGluZGV4W3BdWyJib3hlcyJdOgogICAgICAgICAgICBpbnN0W2luZGV4W3BdWyJzcmMiXV1bYlswXV0gKz0gMQoKICAgICMgc3RyYXRpZmllZCBzcGxpdCBieSBhbmNob3IgPSByYXJlc3QgY2xhc3MgcHJlc2VudCAocG9zdC1kZWR1cGUsIHByZS1zcGxpdCkKICAgIGNsc19jb3VudCA9IGNvbGxlY3Rpb25zLkNvdW50ZXIoKQogICAgZm9yIHAgaW4ga2VlcDoKICAgICAgICBmb3IgYiBpbiBpbmRleFtwXVsiYm94ZXMiXToKICAgICAgICAgICAgY2xzX2NvdW50W2JbMF1dICs9IDEKICAgIGFuY2hvciA9IHt9CiAgICBmb3IgcCBpbiBrZWVwOgogICAgICAgIGNsc19wcmVzZW50ID0gc29ydGVkKHtiWzBdIGZvciBiIGluIGluZGV4W3BdWyJib3hlcyJdfSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBrZXk9bGFtYmRhIGM6IGNsc19jb3VudFtjXSkKICAgICAgICBhbmNob3JbcF0gPSBjbHNfcHJlc2VudFswXSBpZiBjbHNfcHJlc2VudCBlbHNlICJub25lIgoKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oYXJncy5zZWVkKQogICAgZ3JvdXBzID0gY29sbGVjdGlvbnMuZGVmYXVsdGRpY3QobGlzdCkKICAgIGZvciBwIGluIGtlZXA6CiAgICAgICAgZ3JvdXBzW2FuY2hvcltwXV0uYXBwZW5kKHApCiAgICBzdWJzZXQgPSB7InRyYWluIjogW10sICJ2YWwiOiBbXSwgInRlc3QiOiBbXX0KICAgIGZvciBhLCBwcyBpbiBzb3J0ZWQoZ3JvdXBzLml0ZW1zKCkpOgogICAgICAgIHJuZy5zaHVmZmxlKHBzKQogICAgICAgIG4gPSBsZW4ocHMpCiAgICAgICAgbl90ciA9IG1heCgxLCByb3VuZCgwLjggKiBuKSkKICAgICAgICBuX3YgPSBtYXgoMSwgcm91bmQoMC4xICogbikpIGlmIG4gPiAyIGVsc2UgMAogICAgICAgIHN1YnNldFsidHJhaW4iXS5leHRlbmQocHNbOm5fdHJdKQogICAgICAgIHN1YnNldFsidmFsIl0uZXh0ZW5kKHBzW25fdHI6bl90ciArIG5fdl0pCiAgICAgICAgc3Vic2V0WyJ0ZXN0Il0uZXh0ZW5kKHBzW25fdHIgKyBuX3Y6XSkKCiAgICBvdXQgPSBwYXRobGliLlBhdGgoYXJncy5vdXQpCiAgICBmb3Igc3AgaW4gKCJ0cmFpbiIsICJ2YWwiLCAidGVzdCIpOgogICAgICAgIChvdXQgLyAiaW1hZ2VzIiAvIHNwKS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgKG91dCAvICJsYWJlbHMiIC8gc3ApLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBmb3IgcCBpbiBzdWJzZXRbc3BdOgogICAgICAgICAgICBleHQgPSBwLnN1ZmZpeAogICAgICAgICAgICBuZXcgPSBmIntwLnN0ZW1bOjMyXX1fe21kNShwKVs6MTBdfXtleHR9IgogICAgICAgICAgICBzaHV0aWwuY29weShwLCBvdXQgLyAiaW1hZ2VzIiAvIHNwIC8gbmV3KQogICAgICAgICAgICBsaW5lcyA9IFtdCiAgICAgICAgICAgIGltID0gaW5kZXhbcF0KICAgICAgICAgICAgZm9yIGIgaW4gaW1bImJveGVzIl06CiAgICAgICAgICAgICAgICB4MSwgeTEsIHgyLCB5MiA9IGJbMTpdCiAgICAgICAgICAgICAgICB3LCBoID0gaW1bInNpemUiXQogICAgICAgICAgICAgICAgY3ggPSAoeDEgKyB4MikgLyAyIC8gdwogICAgICAgICAgICAgICAgY3kgPSAoeTEgKyB5MikgLyAyIC8gaAogICAgICAgICAgICAgICAgYncgPSAoeDIgLSB4MSkgLyB3CiAgICAgICAgICAgICAgICBiaCA9ICh5MiAtIHkxKSAvIGgKICAgICAgICAgICAgICAgIGxpbmVzLmFwcGVuZCgiICIuam9pbihzdHIodikgZm9yIHYgaW4gKAogICAgICAgICAgICAgICAgICAgIGNsc19uYW1lcy5pbmRleChiWzBdKSwgY3gsIGN5LCBidywgYmgpKSkKICAgICAgICAgICAgIyBsYWJlbHMgd3JpdGUgYWxvbmdzaWRlIGltYWdlcwogICAgICAgICAgICAob3V0IC8gImxhYmVscyIgLyBzcCAvIChwYXRobGliLlBhdGgobmV3KS5zdGVtICsgIi50eHQiKSkud3JpdGVfdGV4dCgKICAgICAgICAgICAgICAgICJcbiIuam9pbihsaW5lcykgKyAoIlxuIiBpZiBsaW5lcyBlbHNlICIiKSkKICAgIChvdXQgLyAiZGF0YS55YW1sIikud3JpdGVfdGV4dCh5YW1sLnNhZmVfZHVtcCgKICAgICAgICB7InBhdGgiOiBzdHIob3V0LnJlc29sdmUoKSksICJ0cmFpbiI6ICJpbWFnZXMvdHJhaW4iLCAidmFsIjogImltYWdlcy92YWwiLAogICAgICAgICAidGVzdCI6ICJpbWFnZXMvdGVzdCIsICJuYyI6IGxlbihjbHNfbmFtZXMpLCAibmFtZXMiOiBjbHNfbmFtZXMsCiAgICAgICAgICJuYW1lc19vcmlnIjogY2xzX25hbWVzfSwgc29ydF9rZXlzPUZhbHNlKSkKCiAgICAjIHJlcG9ydHMKICAgIHRyYWluX2luc3QgPSBjb2xsZWN0aW9ucy5Db3VudGVyKCkKICAgIGZvciBzcCBpbiAoInRyYWluIiwgInZhbCIsICJ0ZXN0Iik6CiAgICAgICAgbHAgPSBvdXQgLyAibGFiZWxzIiAvIHNwCiAgICAgICAgZm9yIGYgaW4gbHAuZ2xvYigiKi50eHQiKToKICAgICAgICAgICAgZm9yIGxpbmUgaW4gZi5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCk6CiAgICAgICAgICAgICAgICB0cmFpbl9pbnN0WyhzcCwgY2xzX25hbWVzW2ludChsaW5lLnNwbGl0KClbMF0pXSldICs9IDEKICAgIHJlcG9ydCA9IHsKICAgICAgICAiaW1hZ2VzX3RvdGFsX3JhdyI6IGltYWdlc19yYXcsCiAgICAgICAgImltYWdlc19hZnRlcl9tZDUiOiBsZW4oaW5kZXgpLAogICAgICAgICJpbWFnZXNfYWZ0ZXJfcGhhc2giOiBsZW4oa2VlcCksCiAgICAgICAgImR1cGxpY2F0ZXNfcmVtb3ZlZCI6IGR1cHMsCiAgICAgICAgImluc3RhbmNlc19wZXJfY2xhc3NfdG90YWwiOiBkaWN0KGNsc19jb3VudCksCiAgICAgICAgImluc3RhbmNlc19wZXJfY2xhc3Nfc3BsaXQiOiB7fSwKICAgICAgICAicmFyZV9jbGFzc2VzX2ZsYWdnZWQiOiBbYyBmb3IgYyBpbiBjbHNfbmFtZXMgaWYKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdHJhaW5faW5zdFsoInRyYWluIiwgYyldIDwgMTUwXSwKICAgICAgICAicGVyX3NvdXJjZV9pbnN0YW5jZXMiOiB7czogZGljdCh2KSBmb3IgcywgdiBpbiBpbnN0Lml0ZW1zKCl9LAogICAgICAgICJzcGxpdF9jb3VudHMiOiB7c3A6IGxlbihzdWJzZXRbc3BdKSBmb3Igc3AgaW4gc3Vic2V0fSwKICAgIH0KICAgIGZvciBzcCBpbiAoInRyYWluIiwgInZhbCIsICJ0ZXN0Iik6CiAgICAgICAgcmVwb3J0WyJpbnN0YW5jZXNfcGVyX2NsYXNzX3NwbGl0Il1bc3BdID0gewogICAgICAgICAgICBjOiB0cmFpbl9pbnN0WyhzcCwgYyldIGZvciBjIGluIGNsc19uYW1lc30KCiAgICAjIG92ZXJzYW1wbGUgcmFyZSBjbGFzc2VzICh0YXNrIHNwZWMpOiBkdXBsaWNhdGUgdHJhaW4gaW1hZ2VzIHRoYXQgY29udGFpbgogICAgIyBhbnkgY2xhc3MgYmVsb3cgLS1vdmVyc2FtcGxlLW1pbiB0cmFpbiBpbnN0YW5jZXMKICAgIGR1cF9jb3VudCA9IDAKICAgIGlmIGFyZ3Mub3ZlcnNhbXBsZV9taW4gPiAwOgogICAgICAgIHJhcmUgPSBbYyBmb3IgYyBpbiBjbHNfbmFtZXMgaWYgdHJhaW5faW5zdFsoInRyYWluIiwgYyldIDwgYXJncy5vdmVyc2FtcGxlX21pbl0KICAgICAgICBpbWdzID0gc29ydGVkKChvdXQgLyAiaW1hZ2VzIiAvICJ0cmFpbiIpLmdsb2IoIioiKSkKICAgICAgICBybmcuc2h1ZmZsZShpbWdzKQogICAgICAgIGZvciBjIGluIHJhcmU6CiAgICAgICAgICAgIG5lZWQgPSBhcmdzLm92ZXJzYW1wbGVfbWluIC0gdHJhaW5faW5zdFsoInRyYWluIiwgYyldCiAgICAgICAgICAgIGZvciBpbWdfcGF0aCBpbiBpbWdzOgogICAgICAgICAgICAgICAgaWYgbmVlZCA8PSAwOgogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBsYmwgPSAob3V0IC8gImxhYmVscyIgLyAidHJhaW4iIC8KICAgICAgICAgICAgICAgICAgICAgICAoaW1nX3BhdGguc3RlbSArICIudHh0IikpCiAgICAgICAgICAgICAgICBpZiBub3QgbGJsLmV4aXN0cygpOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBoYXZlID0gc3VtKDEgZm9yIGxpbmUgaW4gbGJsLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKQogICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBsaW5lLnNwbGl0KCkgYW5kIGNsc19uYW1lc1tpbnQobGluZS5zcGxpdCgpWzBdKV0gPT0gYykKICAgICAgICAgICAgICAgIGlmIGhhdmUgPT0gMDoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgc3RlbSwgZXh0ID0gaW1nX3BhdGguc3RlbSwgaW1nX3BhdGguc3VmZml4CiAgICAgICAgICAgICAgICBzaHV0aWwuY29weShpbWdfcGF0aCwgaW1nX3BhdGgud2l0aF9uYW1lKGYie3N0ZW19X2R1cHtjfXtleHR9IikpCiAgICAgICAgICAgICAgICBzaHV0aWwuY29weShsYmwsIGxibC53aXRoX25hbWUoZiJ7bGJsLnN0ZW19X2R1cHtjfS50eHQiKSkKICAgICAgICAgICAgICAgIGR1cF9jb3VudCArPSAxCiAgICAgICAgICAgICAgICBuZWVkIC09IGhhdmUKICAgICAgICAgICAgICAgIGltZ3MuYXBwZW5kKGltZ19wYXRoLndpdGhfbmFtZShmIntzdGVtfV9kdXB7Y317ZXh0fSIpKQogICAgICAgIGlmIHJhcmU6CiAgICAgICAgICAgIHJlcG9ydFsib3ZlcnNhbXBsZSJdID0geyJjbGFzc2VzIjogcmFyZSwgImltYWdlc19kdXBsaWNhdGVkIjogZHVwX2NvdW50fQoKICAgIChvdXQgLyAiZGF0YXNldF9yZXBvcnQuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0yKSkKICAgIHByaW50KGpzb24uZHVtcHMocmVwb3J0LCBpbmRlbnQ9MikpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=",
    "classes.yaml": "IyBtbC9zdHJlZXQgY2xhc3MgbWFwcGluZyDigJQgdGFyZ2V0IGRldGVjdG9yIGNsYXNzZXMKIyBUYXJnZXQ6IDUgY2xhc3NlcyBmb3IgdGhlIGFjY2Vzc2liaWxpdHktYXdhcmUgcm91dGVyLgpmaW5kaW5nczogICMgaHVtYW4tcmVhZGFibGUgbm90ZXMgZm9yIEZJTkFMX1JFUE9SVAogIC0gInJhbXAgKFplbm9kbykgaGFzIG9ubHkgMTQzIGluc3RhbmNlcyAtPiBiZWxvdyB0aGUgMTUwLWluc3RhbmNlIHZpc2liaWxpdHkgdGhyZXNob2xkIgogIC0gImdyYWJfYmFyIChaZW5vZG8pIGRyb3BwZWQ6IG5vdCB1c2VkIGJ5IHRoZSByb3V0ZXIiCiAgLSAicGVkZXN0cmlhbiBzaWduIC8gc3RvcCBzaWduIC8gdHJhZmZpYyBsaWdodCBwb3N0IC8gYnV0dG9uIC8gc2lnbmFsOiBkcm9wcGVkIChub3QgYmFycmllcnMpIgogIC0gImJhY2tzbGFzaCAoY29tYmluZWQtcHJvaiB2MiBleHRyYSBjbGFzcyk6IGRyb3BwZWQgKG5vIHNlbWFudGljIG1hcHBpbmc7IG1hbGZvcm1lZCBjbGFzcyBuYW1lKSIKY2xhc3NlczoKICAtIG5hbWU6IGN1cmJfcmFtcAogICAgc291cmNlczogICAgICAgICAgICAgICAgIyBzb3VyY2UgY2xhc3MgbmFtZSAtPiBoYW5kbGluZwogICAgICBzaWRld2Fsa3NpZGV3YWxrX3YxMDogImN1cmIgcmFtcCIKICAgICAgbG9nYW5faW50ZXJzZWN0aW9uc192MTogWyJ0ZXh0dXJlZCBjdXJiIHJhbXAiLCAibm9uIHRleHR1cmVkIGN1cmIgcmFtcCJdCiAgICAgIGNvbWJpbmVkX3Byb2pfdjI6ICIxIGN1cmIgcmFtcCIKICAtIG5hbWU6IGNyb3Nzd2FsawogICAgc291cmNlczoKICAgICAgc2lkZXdhbGtzaWRld2Fsa192MTA6ICJ6ZWJyYSBjcm9zc2luZyIKICAgICAgbG9nYW5faW50ZXJzZWN0aW9uc192MTogImNyb3NzIHdhbGsiCiAgICAgIGNvbWJpbmVkX3Byb2pfdjI6ICIwIGNyb3NzIHdhbGsiCiAgLSBuYW1lOiBzdGFpcnNfc3RlcHMKICAgIHNvdXJjZXM6CiAgICAgIHdoZWVsbWFwOiBbInN0ZXAiLCAic3RhaXIiXSAgICMgWmVub2RvIHN0ZXBzICsgc3RhaXJzIGNvbnNvbGlkYXRlZAogIC0gbmFtZTogdGFjdGlsZV9wYXZpbmcKICAgIHNvdXJjZXM6CiAgICAgIHNpZGV3YWxrc2lkZXdhbGtfdjEwOiAiYmxpbmQgdHJhY2tzIgogICAgICBjb21iaW5lZF9wcm9qX3YyOiAiNCB0ZXh0dXJlZCBwYXZpbmciCiAgLSBuYW1lOiByYW1wCiAgICBzb3VyY2VzOgogICAgICB3aGVlbG1hcDogInJhbXAiICAgIyBOT1QgY3VyYl9yYW1wIChzcGVjOiByZXVzZSBaZW5vZG8gcmFtcHMgYXMgcmFtcCkKZHJvcHBlZDoKICB3aGVlbG1hcDogWyJncmFiX2JhciJdCiAgbG9nYW5faW50ZXJzZWN0aW9uc192MTogWyJwZWRlc3RyaWFuIHNpZ24iLCAic3RvcCBzaWduIiwgInRyYWZmaWMgbGlnaHQgcG9zdCJdCiAgY29tYmluZWRfcHJval92MjogWyIyIHBlZGVzdHJpYW4gc2lnbiIsICIzIHN0b3Agc2lnbiIsICI1IHRyYWZmaWMgbGlnaHQgcG9zdCIsICI2IGJ1dHRvbiIsICI3IHNpZ25hbCIsICJcXCJdCiAgc2lkZXdhbGtzaWRld2Fsa192MTA6IFtdCmltYWdlX3NvdXJjZXM6CiAgc2lkZXdhbGtzaWRld2Fsa192MTA6IFJvYm9mbG93IHNpZGV3YWxrZGV0ZWN0Ly1zaWRld2FsayB2MTAgKE5PIGF1Z21lbnRhdGlvbiBleHBvcnQpCiAgbG9nYW5faW50ZXJzZWN0aW9uc192MTogUm9ib2Zsb3cgZXhwZXJpbWVudHMteXB4Z3EvbG9nYW4taW50ZXJzZWN0aW9ucyB2MSAoc3Vic3RpdHV0ZSBmb3IgbG9nYW5pbnRlcnNlY3Rpb24yIHdoaWNoIGhhcyAwIGV4cG9ydGFibGUgdmVyc2lvbnMpCiAgY29tYmluZWRfcHJval92MjogUm9ib2Zsb3cgY29tcHV0ZXJ2aXNpb24tazZ5cGkvY29tYmluZWQtcHJvaiB2MiAoc3Vic3RpdHV0ZSBmb3IgdGVzdC1iNHpobyB3aGljaCBoYXMgMCBleHBvcnRhYmxlIHZlcnNpb25zKQogIHdoZWVsbWFwOiBaZW5vZG8gNjM4MjA5MCAiSW1hZ2UgRGF0YXNldCBvZiBBY2Nlc3NpYmlsaXR5IEJhcnJpZXJzIiAoQ1ZBVCBYTUwgKyA1NTM4IGpwZ3M7IGltYWdlcyB3aXRoIG9ubHkgZ3JhYl9iYXIgYm94ZXMgZXhjbHVkZWQpCg==",
}
for fname, raw in SCRIPT_B64.items():
    if fname.endswith(".py"):
        dst = pathlib.Path("backend/ml/street/scripts", fname)
    else:
        dst = pathlib.Path("backend/ml/street/configs", fname)
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_bytes(base64.b64decode(raw))
print("embedded build_dataset.py + classes.yaml written")



## 1. Fetch datasets

* 3 Roboflow YOLOv8 exports (versions with **zero augmentation**) via API.
* Wheelmap Zenodo record 6382090 — single 7.97 GB zip; we fetch only the
  subset of images carrying `step`/`stair`/`ramp` boxes (~4.4k) by ranged
  HTTP, using zip64-aware central-directory offsets. ~25–40 min.


In [ ]:

import json, os, subprocess

jobs = [
    ("sidewalkdetect/-sidewalk/10", "sidewalksidewalk_v10"),
    ("experiments-ypxgq/logan-intersections/1", "logan_intersections_v1"),
    ("computervision-k6ypi/combined-proj/2", "combined_proj_v2"),
]
for ep, nm in jobs:
    dst = f"backend/ml/street/data/{nm}"
    if os.path.exists(f"{dst}/data.yaml"):
        print(nm, "already present")
        continue
    out = subprocess.run(
        ["curl", "-s", f"https://api.roboflow.com/{ep}/yolov8?download=true&api_key={os.environ['ROBOFLOW_API_KEY']}"],
        capture_output=True, text=True).stdout
    d = json.loads(out)
    link = (d.get("export") or {}).get("link")
    assert link, (nm, out[:200])
    print(nm, "-> exporting")
    !curl -sfL "{link}" -o "{nm}.zip"
    !unzip -q -o "{nm}.zip" -d "backend/ml/street/data/{nm}" && rm -f "{nm}.zip"
    print(nm, "extracted")
print("roboflow ready")


In [ ]:

import os, pathlib, struct, subprocess, xml.etree.ElementTree as ET, zlib, collections

WM = "backend/ml/street/data/wheelmap"
pathlib.Path(f"{WM}/images").mkdir(parents=True, exist_ok=True)
URL = "https://zenodo.org/api/records/6382090/files/wm_barriers_data.zip/content"

def fetch(start, end):
    return bytes(subprocess.run(["curl", "-sf", "-r", f"{start}-{end}", URL],
                                capture_output=True).stdout)

# -- first archive member = wm_annotations.xml (deflate), parse its local header
buf = fetch(0, 393215)
assert buf[:4] == b"PK\x03\x04"
nl, el = struct.unpack_from("<HH", buf, 26)
csize = struct.unpack_from("<I", buf, 18)[0]
xml = zlib.decompressobj(-15).decompress(buf[30 + nl + el: 30 + nl + el + csize]).decode()
open(f"{WM}/wm_annotations.xml", "w").write(xml)
print("wm_annotations.xml:", len(xml), "bytes (csize", csize, ")")

# -- keep images that contain >=1 step/stair/ramp box (CVAT XML 1.1)
keep, counts = set(), collections.Counter()
for im in ET.parse(f"{WM}/wm_annotations.xml").getroot().findall("image"):
    if any(b.attrib["label"] in ("step", "stair", "ramp") for b in im.findall("box")):
        keep.add(im.attrib["name"])
        for b in im.findall("box"):
            counts[b.attrib["label"]] += 1
print("images to keep:", len(keep), dict(counts))


In [ ]:

# -- zip64-aware central directory parse -> (name, csize, local_header_offset) rows
import struct, json

total = 0
for line in subprocess.run(["curl", "-sI", URL], capture_output=True, text=True).stdout.splitlines():
    if line.lower().startswith("content-length:"):
        total = int(line.split(":", 1)[1].strip())
assert total > 0, "no Content-Length"
print("zip size:", total)

eocd_tail = fetch(total - 65536, total - 1)
i = eocd_tail.rfind(b"PK\x05\x06")
assert i >= 0, "EOCD not found in last 64 KiB"
# EOCD: eocd+i+8 = (this-disk H, total-entries H, cd-size I, cd-offset I)
_, n_e, cd_size, cd_ofst = struct.unpack_from("<HHII", eocd_tail, i + 8)
if 0xFFFFFFFF in (n_e, cd_size, cd_ofst):  # zip64 EOCD locator + EOCD64
    j = eocd_tail.rfind(b"PK\x06\x07", 0, i)
    assert j >= 0, "zip64 locator not found"
    k = eocd_tail.rfind(b"PK\x06\x06", 0, j)
    assert k >= 0, "EOCD64 record not found"
    # EOCD64: sig 4s, recsize Q, ver H, ver H, disk I, cd-disk I,
    #         entries-on-disk Q, total-entries Q, cd-size Q, cd-offset Q
    _, _, _, _, _, _, _, ntotal, cdsz64, eof = struct.unpack_from("<4sQHHIIQQQQ", eocd_tail, k)
    n_e, cd_size, cd_ofst = ntotal, cdsz64, eof
print("cd entries:", n_e, "| cd at", cd_ofst, "size", cd_size)

cd = fetch(cd_ofst, cd_ofst + cd_size - 1)
rows, pos = [], 0
while pos < len(cd) - 4 and cd[pos:pos + 4] == b"PK\x01\x02":
    nl, el, clen = struct.unpack_from("<HHH", cd, pos + 28)
    name = cd[pos + 46: pos + 46 + nl].decode()
    csize = struct.unpack_from("<I", cd, pos + 20)[0]
    lho = struct.unpack_from("<I", cd, pos + 42)[0]
    extra = cd[pos + 46 + nl: pos + 46 + nl + el]
    if lho == 0xFFFFFFFF or csize == 0xFFFFFFFF:  # zip64 extra field, eid 0x0001
        vals, p2 = [], 0
        while p2 < len(extra) - 3:
            eid, esz = struct.unpack_from("<HH", extra, p2)
            if eid == 0x0001:
                q = p2 + 4
                while q + 8 <= p2 + 4 + esz:
                    vals.append(struct.unpack_from("<Q", extra, q)[0])
                    q += 8
                break
            p2 += 4 + esz
        if csize == 0xFFFFFFFF:
            csize = vals.pop(0)
        if lho == 0xFFFFFFFF:
            lho = vals[-1]
    if name.startswith("images/") and name in keep:
        rows.append((name, csize, lho))
        assert 0 <= lho < total, (name, lho)
    pos += 46 + nl + el + clen
print("wanted rows:", len(rows))
json.dump(rows, open("/tmp/zen_wanted.json", "w"))


In [ ]:

# -- parallel ranged member extraction (resumable, retry w/ backoff)
import concurrent.futures as cf, pathlib, struct, time, zlib

rows = json.load(open("/tmp/zen_wanted.json"))
OUT = pathlib.Path("backend/ml/street/data/wheelmap/images")

def one(row):
    name, csize, lho = row
    p = OUT / pathlib.PurePosixPath(name).name
    if p.exists() and p.stat().st_size > 0:
        return "cached"
    end = lho + 4096 + csize
    for attempt in range(4):
        try:
            buf = fetch(lho, end - 1)
            if buf[:4] != b"PK\x03\x04":
                raise ValueError("bad local hdr")
            nl, el = struct.unpack_from("<HH", buf, 26)
            raw = zlib.decompressobj(-15).decompress(buf[30 + nl + el: 30 + nl + el + csize])
            p.write_bytes(raw)
            return "ok"
        except Exception:
            time.sleep(2 + 2 * attempt)
    return "FAIL"

t0 = time.time()
fails = 0
with cf.ThreadPoolExecutor(8) as ex:
    for idx, st in enumerate(ex.map(one, rows)):
        if st == "FAIL":
            fails += 1
        if idx % 400 == 0:
            print(f"{idx}/{len(rows)} elapsed {time.time()-t0:.0f}s", flush=True)
print("fails:", fails)


In [ ]:

# retry rounds until zero fails
import os
for rnd in range(4):
    n_fail = sum(1 for st in cf.ThreadPoolExecutor(8).map(one, rows) if st == "FAIL")
    print("round", rnd, "fails:", n_fail)
    if n_fail == 0:
        break
print("images on disk:", len(os.listdir(OUT)))
assert len(os.listdir(OUT)) >= len(rows) - 5, "fetch incomplete"



## 2. Build the unified YOLO dataset

Embedded `build_dataset.py`: class mapping (curb ramps→`curb_ramp`,
crossings→`crosswalk`, blind tracks/textured paving→`tactile_paving`,
Wheelmap step+stair→`stairs_steps`, ramp→`ramp`), MD5 exact + perceptual
(pHash, hamming ≤ 4) dedupe **before** splitting, stratified 80/10/10,
rare-class (<150 train instances) oversampling.


In [ ]:

import runpy, sys
sys.argv = ["build_dataset.py"]  # jupyter/colab kernel leaves -f runtime.json in sys.argv
runpy.run_path("backend/ml/street/scripts/build_dataset.py", run_name="__main__")



## 3. Train — YOLO11s, imgsz 640, ≤100 epochs, patience 20

Never tune on test.


In [ ]:

import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print("device:", device)
model = YOLO("yolo11s.pt")
model.train(
    data="backend/ml/street/data/yolo/data.yaml",
    epochs=100, patience=20, imgsz=640, batch=16,
    device=device, project="backend/ml/street/runs",
    name="yolo11s_street", exist_ok=True,
)
print("done -> backend/ml/street/runs/yolo11s_street/weights/best.pt")



## 4. Held-out evaluation (val + test) → `metrics/street.json`

Test is touched exactly once, for reporting.


In [ ]:

import json, pathlib, shutil
from ultralytics import YOLO

w = "backend/ml/street/runs/yolo11s_street/weights/best.pt"
shutil.copy(w, "backend/ml/street/weights/best.pt")
model = YOLO(w)

out = {}
for split in ("val", "test"):
    r = model.val(data="backend/ml/street/data/yolo/data.yaml", split=split,
                  imgsz=640, device=device, plots=True,
                  project="backend/ml/street/runs", name=f"eval_{split}",
                  exist_ok=True, save_json=False, verbose=False)
    per = {}
    for i, name in model.names.items():
        try:
            per[name] = {"P": round(float(r.box.p[i]), 4),
                         "R": round(float(r.box.r[i]), 4),
                         "mAP50": round(float(r.box.ap50[i]), 4),
                         "mAP50-95": round(float(r.box.ap[i]), 4)}
        except IndexError:
            pass
    out[split] = {"overall": {k: round(float(v), 4) for k, v in r.results_dict.items()},
                  "per_class": per}
    print(split, "->", json.dumps(out[split]["overall"], indent=1))
pathlib.Path("backend/ml/street/metrics").mkdir(exist_ok=True)
pathlib.Path("backend/ml/street/metrics/street.json").write_text(json.dumps(out, indent=2))
print("street.json written")



## 5. `detect()` contract check + 5 annotated samples + ONNX export


In [ ]:

import glob, io, pathlib, shutil, sys
from PIL import Image, ImageDraw
from ultralytics import YOLO

WEIGHTS = "backend/ml/street/weights/best.pt"
model = YOLO(WEIGHTS)

def detect(image_bytes, conf=0.25):
    """Contract: bytes -> [{class, confidence, bbox:[x1,y1,x2,y2]}]."""
    results = model.predict(Image.open(io.BytesIO(image_bytes)).convert("RGB"),
                            conf=conf, verbose=False)
    names = results[0].names
    return [{"class": names[int(b.cls.item())],
             "confidence": round(float(b.conf.item()), 4),
             "bbox": [round(float(v), 1) for v in b.xyxy[0].tolist()]}
            for b in results[0].boxes]

samples = sorted(glob.glob("backend/ml/street/data/yolo/images/test/*"))[:5]
for s in samples:
    img_raw = open(s, "rb").read()
    det = detect(img_raw, conf=0.25)
    im = Image.open(s).convert("RGB")
    dr = ImageDraw.Draw(im)
    for d in det:
        x1, y1, x2, y2 = d["bbox"]
        dr.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=3)
        dr.text((x1, max(0, y1 - 12)), f"{d['class']} {d['confidence']:.2f}", fill=(255, 0, 0))
    outp = pathlib.Path("backend/ml/street/samples") / (pathlib.Path(s).stem + "_annotated.jpg")
    im.save(outp, quality=88)
    print(pathlib.Path(s).name, "->", len(det), "detections,", outp.name)

onnx_path = model.export(format="onnx")
shutil.copy(onnx_path, "backend/ml/street/weights/best.onnx")
print("onnx ready:", "backend/ml/street/weights/best.onnx")



## 6. Payload zip → download → drop into repo


In [ ]:

import subprocess

sp = subprocess.run(["zip", "-r", "weights_payload.zip",
                     "backend/ml/street/weights/best.pt",
                     "backend/ml/street/weights/best.onnx",
                     "backend/ml/street/metrics/street.json",
                     "backend/ml/street/samples",
                     "backend/ml/street/data/yolo/dataset_report.json"],
                    capture_output=True, text=True)
print(sp.stdout[-400:], sp.stderr[-400:])
try:
    from google.colab import files
    files.download("weights_payload.zip")
except Exception as e:
    print("auto-download unavailable:", e)
